# Lesson 04 - ਟੂਲ ਵਰਤੋਂ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੇਂਟ ਫਰੇਮਵਰਕ (Python) ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹੋਏ AI ਏਜੰਟਾਂ ਲਈ **ਟੂਲ ਵਰਤੋਂ** ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਸਿੱਖੋਗੇ। ਅਸੀਂ ਕਵਰ ਕਰਾਂਗੇ:

- `@tool` ਡੈੱਕੋਰੇਟਰ ਅਤੇ ਟਾਈਪ ਕੀਤੇ ਪੈਰਾਮੀਟਰਾਂ ਨਾਲ ਫੰਕਸ਼ਨ ਟੂਲਾਂ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨਾ
- ਮਾਡਲ ਨੂੰ ਪਤਾ ਹੋਵੇ ਕਿ ਹਰ ਇੱਕ ਟੂਲ ਕੀ ਕਰਦਾ ਹੈ, ਇਸ ਲਈ ਟੂਲ ਸਕੀਮਾਂ ਪ੍ਰਦਾਨ ਕਰਨਾ
- `approval_mode` ਨਾਲ ਟੂਲ ਦੇ ਚਲਾਉਣ 'ਤੇ ਕੰਟਰੋਲ ਕਰਨਾ
- Pydantic ਮਾਡਲਾਂ ਅਤੇ `response_format` ਰਾਹੀਂ **ਸੰਰਚਿਤ ਨਤੀਜਾ** ਵਾਪਸ ਕਰਨਾ

ਦ੍ਰਿਸ਼ਟਾਂਤ ਇੱਕ **ਸਫ਼ਰ ਬੁਕਿੰਗ ਏਜੰਟ** ਹੈ ਜੋ ਮੰਜ਼ਿਲਾਂ ਦੀ ਜਾਂਚ ਕਰ ਸਕਦਾ ਹੈ, ਉਪਲਬਧਤਾ ਵੇਖ ਸਕਦਾ ਹੈ, ਅਤੇ ਫਲਾਈਟ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰ ਸਕਦਾ ਹੈ।


## ਸੈੱਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ਡੀਕੋਰੇਟਰ ਨਾਲ ਟੂਲਜ਼ ਦੀ ਪਰਿਭਾਸ਼ਾ

`@tool` ਡੀਕੋਰੇਟਰ ਇੱਕ ਸਧਾਰਨ Python ਫੰਕਸ਼ਨ ਨੂੰ ਇੱਕ ਟੂਲ ਵਿੱਚ ਬਦਲ ਦਿੰਦਾ ਹੈ ਜਿਸਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।  
ਮੁੱਖ ਬਿੰਦੂ:

- **ਡੌਕਸਟਰਿੰਗ** ਟੂਲ ਦਾ ਵੇਰਵਾ ਬਣ ਜਾਂਦੀ ਹੈ ਜੋ ਮਾਡਲ ਵੇਖਦਾ ਹੈ।  
- **ਟਾਈਪ ਐਨੋਟੇਸ਼ਨਜ਼** (ਜਿਨ੍ਹਾਂ ਵਿੱਚ ਵੇਰਵੇ ਦੇ ਨਾਲ `Annotated` ਸ਼ਾਮਲ ਹੈ) ਟੂਲ ਦਾ ਸਕੀਮਾ ਨਿਰਧਾਰਤ ਕਰਦੇ ਹਨ।  
- `approval_mode` ਇਹ ਨਿਰਣੈ ਕਰਦਾ ਹੈ ਕਿ ਕੀਤਾ ਹਰ ਕਾਲ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਯੂਜ਼ਰ ਦੀ ਮਨਜ਼ੂਰੀ ਲਾਜ਼ਮੀ ਹੈ ਜਾਂ ਨਹੀਂ।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ਇੱਕ ਏਜੰਟ ਬਹੁਤ ਸਾਰੇ ਟੂਲਾਂ ਨਾਲ ਬਣਾਉਣਾ

ਮਾਡਲ ਲੈ ਸਕੇ ਅਜਿਹੇ ਕਿਸੇ ਵੀ ਟੂਲ ਨੂੰ ਜਿਹੜਾ ਉਹਨਾਂ ਨੂੰ ਵਰਤ ਕੇ ਯੂਜ਼ਰ ਦੇ ਸਵਾਲ ਦਾ ਜਵਾਬ ਦੇ ਸਕੇ ਲਈ ਸਾਰੇ ਤਿੰਨ ਟੂਲਾਂ ਨੂੰ ਕਲਾਇੰਟ ਨੂੰ ਦੇ ਦਿਓ।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Structured Output with Tools

By setting `response_format` to a Pydantic model, the agent is forced to return a well-typed JSON object instead of free-form text. This is useful when downstream code needs to consume the result programmatically.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ਟੂਲ ਮਨਜ਼ੂਰੀ ਦੇ ਪੈਟਰਨ

`@tool` 'ਤੇ `approval_mode` ਪੈਰामीਟਰ ਨਿਯੰਤਰਿਤ ਕਰਦਾ ਹੈ ਕਿ ਟੂਲ ਕਾਲਾਂ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਮਨੁੱਖੀ ਮਨਜ਼ੂਰੀ ਦੀ ਲੋੜ ਹੈ ਜਾਂ ਨਹੀਂ:

| ਮੋਡ | ਵਰਤਾਰਾ |
|---|---|
| `"never_require"` | ਟੂਲ ਆਪੋآپ ਚੱਲਦਾ ਹੈ — ਕਿਸੇ ਉਪਭੋਗਤਾ ਦੀ ਪੁਸ਼ਟੀ ਦੀ ਲੋੜ ਨਹੀਂ। |
| `"always_require"` | ਹਰ ਕਾਲ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਉਪਭੋਗਤਾ ਵੱਲੋਂ ਮਨਜ਼ੂਰੀ ਲੈਣੀ ਚਾਹੀਦੀ ਹੈ। |

ਉਹਨਾਂ ਟੂਲਾਂ ਲਈ `"always_require"` ਵਰਤੋਂ ਜਿਨ੍ਹਾਂ ਦੇ ਸਾਈਡ-ਇਫੈਕਟ ਹੁੰਦੇ ਹਨ (ਜਿਵੇਂ ਕਿ ਫਲਾਈਟ ਬੁਕ ਕਰਨਾ, ਕ੍ਰੈਡਿਟ ਕਾਰਡ ਚਾਰਜ ਕਰਨਾ) ਤਾਂ ਜੋ ਮਨੁੱਖ ਸਿੱਕੇ ਵਿੱਚ ਰਿਹਾ ਜਾ ਸਕੇ।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ ਕਿਵੇਂ:

1. ਟਾਈਪ ਕੀਤੇ ਪੈਰਾਮੀਟਰਾਂ ਅਤੇ ਡੌਕਸਟ੍ਰਿੰਗਜ਼ ਵਾਲੇ `@tool` ਡੈਕਾਰੇਟਰ ਦੀ ਵਰਤੋਂ ਕਰਕੇ **ਟੂਲ ਪਰਿਭਾਸ਼ਿਤ ਕਰੀਏ ਜੋ ਟੂਲ ਸਕੀਮਾ ਵਜੋਂ ਕੰਮ ਕਰਦੇ ਹਨ**।
2. ਕਈ ਟੂਲਾਂ ਨੂੰ **ਇਕੱਠਾ ਜੋੜਨਾ ਤਾਂ ਜੋ ਏਜੰਟ ਸੀਕੁਐਂਸ ਵਿੱਚ ਉਹਨਾਂ ਨੂੰ ਕਾਲ ਕਰ ਸਕੇ ਅਤੇ ਜਟਿਲ ਪ੍ਰਸ਼ਨਾਂ ਦੇ ਜਵਾਬ ਦੇ ਸਕੇ**।
3. ਪਾਈਡੈਂਟਿਕ ਮਾਡਲ ਨੂੰ `response_format` ਵਜੋਂ ਪਾਸ ਕਰਕੇ **ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ ਵਾਪਸ ਦੇਣਾ**।
4. `approval_mode` ਨਾਲ **ਟੂਲ ਮਨਜ਼ੂਰੀ ਨੂੰ ਨਿਯੰਤਰਿਤ ਕਰਨਾ ਤਾਂ ਜੋ ਸੰਵੇਦਨਸ਼ੀਲ ਕਾਰਵਾਈਆਂ ਲਈ ਮਨੁੱਖ ਨੂੰ ਸ਼ਾਮਲ ਰੱਖਿਆ ਜਾ ਸਕੇ**।

ਇਹ ਪੈਟਰਨ ਭਰੋਸੇਮੰਦ, ਪ੍ਰੋਡਕਸ਼ਨ-ਤਿਆਰ ਏਜੰਟ ਬਣਾਉਣ ਦੀ ਬੁਨਿਆਦ ਹਨ ਜੋ ਬਾਹਰੀ ਪ੍ਰਣਾਲੀਆਂ ਨਾਲ ਸੁਰੱਖਿਅਤ ਤਰੀਕੇ ਨਾਲ ਇੰਟਰੈਕਟ ਕਰ ਸਕਦੇ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
